# Linear OT Embedding and Tangent Coordinates

This notebook generates `fig:dualnorms-linear-ot-embedding`.  Linear OT fixes an absolutely continuous reference law $\rho$ and represents each target law $\alpha$ by the displacement field
$$
    T_\alpha-\mathrm{Id}\in L^2(\rho),
$$
where $T_\alpha{}_{\sharp}\rho=\alpha$.  In one dimension this is exact because quantile functions linearize $\mathcal W_2$.  In higher dimension the same construction is a tangent approximation: averaging maps is cheap and informative, but it is not the true nonlinear Wasserstein barycenter in general.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import ot

ROOT = Path.cwd()
if not (ROOT / "notebooks-figures").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "notebooks-figures"))

from figure_style import (
    RED, BLUE, VIOLET, ORANGE, GRAY, LIGHT_GRAY,
    DIRAC_MARKER_SIZE, setup_matplotlib, figure_dir, save_pdf,
    remove_axes, box_axes, interp_color, padded_limits,
)

setup_matplotlib()

NAME = "dualnorms-linear-ot-embedding"
OUT = figure_dir(NAME)


## One-dimensional exact quantile linearization

For a one-dimensional reference distribution, the Brenier map is the monotone rearrangement $T_\alpha=Q_\alpha\circ F_\rho$.  Averaging the displacements $T_\alpha-\mathrm{Id}$ therefore coincides with averaging quantile functions, hence gives the exact Wasserstein barycenter.

In [ ]:
def gaussian(x, m, s):
    return np.exp(-0.5*((x-m)/s)**2) / (np.sqrt(2*np.pi)*s)


def normalize_density(x, rho):
    dx = x[1] - x[0]
    return rho / (rho.sum() * dx)


def quantile_from_density(x, rho, r):
    dx = x[1] - x[0]
    F = np.cumsum(rho) * dx
    F = np.maximum.accumulate(F)
    F[-1] = 1.0
    return np.interp(r, F, x)

x = np.linspace(-3.8, 3.8, 1500)
r = np.linspace(1e-4, 1 - 1e-4, 820)
rho_ref = normalize_density(x, gaussian(x, 0.0, 0.78))
rho_a = normalize_density(x, 0.52*gaussian(x, -1.20, 0.34) + 0.48*gaussian(x, 0.30, 0.46))
rho_b = normalize_density(x, 0.38*gaussian(x, -0.20, 0.30) + 0.62*gaussian(x, 1.42, 0.48))
Qref = quantile_from_density(x, rho_ref, r)
Qa = quantile_from_density(x, rho_a, r)
Qb = quantile_from_density(x, rho_b, r)
Da = Qa - Qref
Db = Qb - Qref
Dmid = 0.5 * (Da + Db)
Qmid = Qref + Dmid
rho_mid_at_q = 1.0 / np.maximum(np.gradient(Qmid, r), 1e-8)


## Two-dimensional discrete maps

For the two-dimensional display we use small equal-weight point clouds.  POT solves the quadratic transport plan from the central Gaussian reference to two target mixtures placed respectively on the left and on the right.  The barycentric projections of these plans give two discrete maps $T_0$ and $T_1$ from the same reference sample.

In [ ]:
rng = np.random.default_rng(811)
n = 82
reference = rng.normal(loc=(0.0, 0.0), scale=(0.26, 0.25), size=(n, 2))


def sample_mixture(centers, scales, counts):
    parts = []
    for c, s, k in zip(centers, scales, counts):
        parts.append(rng.normal(loc=c, scale=s, size=(k, 2)))
    return np.vstack(parts)

left_target = sample_mixture(
    centers=np.array([[-1.25, 0.55], [-1.44, -0.42], [-0.88, -0.05]]),
    scales=np.array([[0.16, 0.13], [0.14, 0.17], [0.13, 0.12]]),
    counts=[27, 28, 27],
)
right_target = sample_mixture(
    centers=np.array([[1.20, 0.54], [1.44, -0.44], [0.86, 0.02]]),
    scales=np.array([[0.16, 0.13], [0.14, 0.17], [0.13, 0.12]]),
    counts=[27, 28, 27],
)

a = np.full(n, 1.0 / n)
b = np.full(n, 1.0 / n)
C_left = ot.dist(reference, left_target, metric="sqeuclidean")
C_right = ot.dist(reference, right_target, metric="sqeuclidean")
P_left = ot.emd(a, b, C_left)
P_right = ot.emd(a, b, C_right)
T_left = n * P_left @ left_target
T_right = n * P_right @ right_target
D_left = T_left - reference
D_right = T_right - reference

all_points = np.vstack([reference, left_target, right_target, T_left, T_right])
xlim, ylim = padded_limits(all_points, pad=0.13)


## Exported one-dimensional panels

The first one-dimensional panel displays displacement fields in $L^2(\rho)$.  The second panel displays the densities obtained from the two target quantiles and their averaged quantile.

In [ ]:
fig, ax = plt.subplots(figsize=(2.95, 1.86))
ax.plot(Qref, Da, color=RED, lw=1.15)
ax.plot(Qref, Db, color=BLUE, lw=1.15)
ax.plot(Qref, Dmid, color=VIOLET, lw=1.45)
ax.axhline(0, color=LIGHT_GRAY, lw=0.65)
ax.set_xlim(-2.55, 2.55)
ax.set_ylim(min(Da.min(), Db.min()) - 0.12, max(Da.max(), Db.max()) + 0.12)
ax.tick_params(labelbottom=False, labelleft=False)
box_axes(ax)
save_pdf(fig, OUT / "displacements-1d.pdf", pad_inches=0.055)
plt.close(fig)

fig, ax = plt.subplots(figsize=(2.95, 1.86))
ax.plot(x, rho_ref, color=LIGHT_GRAY, lw=0.90, alpha=0.90)
ax.plot(x, rho_a, color=RED, lw=1.08, alpha=0.90)
ax.plot(x, rho_b, color=BLUE, lw=1.08, alpha=0.90)
ax.plot(Qmid, rho_mid_at_q, color=VIOLET, lw=1.42)
ax.set_xlim(-3.0, 3.0)
ax.set_ylim(0, 0.94)
ax.tick_params(labelbottom=False, labelleft=False)
box_axes(ax)
save_pdf(fig, OUT / "barycenter-1d.pdf", pad_inches=0.055)
plt.close(fig)


## Exported two-dimensional panels

The two map panels show the same central reference sample transported to each side target.  Thin arrows are drawn only on a deterministic subsample so the geometry stays readable.  The final panel overlays four point clouds obtained by linear interpolation of the two displacement fields.

In [ ]:
def finish_cloud_axis(ax, points, pad=0.10):
    local_xlim, local_ylim = padded_limits(points, pad=pad)
    ax.set_xlim(*local_xlim)
    ax.set_ylim(*local_ylim)
    ax.set_aspect("equal")
    remove_axes(ax)


def draw_map_panel(target, mapped, target_color, path):
    fig, ax = plt.subplots(figsize=(2.34, 1.92))
    panel_points = np.vstack([reference, target, mapped])
    ax.scatter(reference[:, 0], reference[:, 1], s=DIRAC_MARKER_SIZE * 0.38, marker="o", color=LIGHT_GRAY, edgecolor="none", linewidth=0, alpha=0.78, zorder=2)
    ax.scatter(target[:, 0], target[:, 1], s=DIRAC_MARKER_SIZE * 0.54, marker="o", color=target_color, edgecolor="none", linewidth=0, alpha=0.78, zorder=3)
    order = np.argsort(np.linalg.norm(mapped - reference, axis=1))[::-1][::3]
    for i in order[:26]:
        ax.annotate(
            "",
            xy=mapped[i],
            xytext=reference[i],
            arrowprops=dict(arrowstyle="-|>", color=VIOLET, lw=0.52, alpha=0.50, shrinkA=1.0, shrinkB=1.0, mutation_scale=5.6),
            zorder=4,
        )
    ax.scatter(mapped[:, 0], mapped[:, 1], s=DIRAC_MARKER_SIZE * 0.34, marker="o", color=VIOLET, edgecolor="none", linewidth=0, alpha=0.55, zorder=5)
    finish_cloud_axis(ax, panel_points, pad=0.10)
    save_pdf(fig, path, pad_inches=0.050)
    plt.close(fig)


draw_map_panel(left_target, T_left, RED, OUT / "map-left-2d.pdf")
draw_map_panel(right_target, T_right, BLUE, OUT / "map-right-2d.pdf")

fig, ax = plt.subplots(figsize=(2.64, 1.92))
trajectory_points = []
for t in [0.0, 0.33, 0.67, 1.0]:
    Z = reference + ((1 - t) * D_left + t * D_right)
    trajectory_points.append(Z)
    ax.scatter(Z[:, 0], Z[:, 1], s=DIRAC_MARKER_SIZE * 0.34, marker="o", color=interp_color(t), edgecolor="none", linewidth=0, alpha=0.62, zorder=2 + int(10*t))
# A few tangent trajectories show that interpolation is performed in map space.
for i in np.argsort(np.linalg.norm(D_right - D_left, axis=1))[::-1][::8][:14]:
    curve = np.stack([reference[i] + ((1 - t) * D_left[i] + t * D_right[i]) for t in np.linspace(0, 1, 30)])
    ax.plot(curve[:, 0], curve[:, 1], color=GRAY, lw=0.38, alpha=0.32, zorder=1)
finish_cloud_axis(ax, np.vstack(trajectory_points), pad=0.08)
save_pdf(fig, OUT / "interpolation-2d.pdf", pad_inches=0.050)
plt.close(fig)
